### Setup Instructions

1. Create a `.env` file in this directory
2. Add your OpenAI API key:

        OPENAI_API_KEY=your_key_here


In [ ]:
from dotenv import load_dotenv
import json
import ast
import os
from datetime import datetime, UTC

from openai import OpenAI


In [ ]:
load_dotenv()

client = OpenAI()


In [83]:
INPUT_PATH = "physician_copilot_student_cases.json"
OUTBOX_PATH = "outbox.json"

def load_cases(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

cases = load_cases(INPUT_PATH)
print(f"Loaded {len(cases)} patient cases.")
cases[0]

Loaded 8 patient cases.


{'patient_id': 'LAB_001',
 'age': 29,
 'sex': 'female',
 'fasting': True,
 'conditions': [],
 'medications': [],
 'symptoms': ['fatigue'],
 'labs': {'LDL': 118,
  'HDL': 62,
  'Triglycerides': 92,
  'A1c': 5.2,
  'Creatinine': 0.8,
  'eGFR': 104,
  'ALT': 20,
  'AST': 19,
  'Vitamin_D': 16}}

In [84]:
# Lab reference ranges are derived from standard clinical guidelines and typical
# laboratory cutoffs and are discretized into clinically meaningful thresholds to enable rule-based
# detection, prioritization, and severity classification within the agent workflow
LAB_RANGES = {
    "LDL": {"borderline": 130, "high": 160, "very_high": 190},
    "HDL": {"low": 40, "good": 60},
    "Triglycerides": {"high": 150, "very_high": 500},
    "A1c": {"prediabetes": 5.7, "diabetes_level": 6.5, "urgent": 10.0},
    "Creatinine": {"high": 1.3, "very_high": 2.0},
    "eGFR": {"low": 60, "very_low": 30},
    "ALT": {"high": 55, "very_high": 150},
    "AST": {"high": 40, "very_high": 150},
    "Vitamin D": {"deficient": 20, "insufficient": 30}
}

In [85]:
def normalize_lab_name(lab):
    return lab.replace("_", " ")

In [86]:
def range_check(lab, value):
    """
    Returns a structured interpretation for a single lab value

    Output fields:
        lab      : normalized lab name
        value    : numeric lab value
        status   : normal / abnormal / missing
        priority : 0 (normal) to 3 (highest concern)
        note     : short interpretation used in downstream prioritization
    """
    lab = normalize_lab_name(lab)

    if value is None:
        return {
            "lab": lab,
            "value": value,
            "status": "missing",
            "priority": 0,
            "note": "No value provided."
        }

    result = {
        "lab": lab,
        "value": value,
        "status": "normal",
        "priority": 0,
        "note": ""
    }

    if lab == "LDL":
        if value >= LAB_RANGES["LDL"]["very_high"]:
            result.update(status="abnormal", priority=3, note="LDL is very elevated.")
        elif value >= LAB_RANGES["LDL"]["high"]:
            result.update(status="abnormal", priority=2, note="LDL is elevated.")
        elif "borderline" in LAB_RANGES["LDL"] and value >= LAB_RANGES["LDL"]["borderline"]:
            result.update(status="abnormal", priority=1, note="LDL is mildly elevated.")

    elif lab == "HDL":
        if value < LAB_RANGES["HDL"]["low"]:
            result.update(status="abnormal", priority=1, note="HDL is low.")

    elif lab == "Triglycerides":
        if value >= LAB_RANGES["Triglycerides"]["very_high"]:
            result.update(status="abnormal", priority=3, note="Triglycerides are very elevated.")
        elif value >= LAB_RANGES["Triglycerides"]["high"]:
            result.update(status="abnormal", priority=2, note="Triglycerides are elevated.")

    elif lab == "A1c":
        if value >= LAB_RANGES["A1c"]["urgent"]:
            result.update(status="abnormal", priority=3, note="A1c is markedly elevated.")
        elif value >= LAB_RANGES["A1c"]["diabetes_level"]:
            result.update(status="abnormal", priority=2, note="A1c is elevated.")
        elif value >= LAB_RANGES["A1c"]["prediabetes"]:
            result.update(status="abnormal", priority=1, note="A1c is mildly elevated.")

    elif lab == "Creatinine":
        if "very_high" in LAB_RANGES["Creatinine"] and value >= LAB_RANGES["Creatinine"]["very_high"]:
            result.update(status="abnormal", priority=3, note="Creatinine is markedly elevated.")
        elif value > LAB_RANGES["Creatinine"]["high"]:
            result.update(status="abnormal", priority=2, note="Creatinine is elevated.")

    elif lab == "eGFR":
        if value < LAB_RANGES["eGFR"]["very_low"]:
            result.update(status="abnormal", priority=3, note="eGFR is significantly reduced.")
        elif value < LAB_RANGES["eGFR"]["low"]:
            result.update(status="abnormal", priority=2, note="eGFR is reduced.")

    elif lab == "ALT":
        if value >= LAB_RANGES["ALT"]["very_high"]:
            result.update(status="abnormal", priority=3, note="ALT is markedly elevated.")
        elif value > LAB_RANGES["ALT"]["high"]:
            result.update(status="abnormal", priority=2, note="ALT is elevated.")

    elif lab == "AST":
        if value >= LAB_RANGES["AST"]["very_high"]:
            result.update(status="abnormal", priority=3, note="AST is markedly elevated.")
        elif value > LAB_RANGES["AST"]["high"]:
            result.update(status="abnormal", priority=2, note="AST is elevated.")

    elif lab == "Vitamin D":
        if value < LAB_RANGES["Vitamin D"]["deficient"]:
            result.update(status="abnormal", priority=1, note="Vitamin D is low.")
        elif value < LAB_RANGES["Vitamin D"]["insufficient"]:
            result.update(status="abnormal", priority=1, note="Vitamin D is slightly low.")

    return result

In [87]:
# Prioritization ranks abnormal labs by clinical importance using rule-based severity scores
# Higher priority values indicate more urgent findings, ensuring that the most clinically significant abnormalities are surfaced first

def prioritize_findings(flagged_labs, top_n=None):
    
    abnormal = [x for x in flagged_labs if x.get("status") == "abnormal"]
    abnormal.sort(key=lambda x: (-x["priority"], x["lab"]))

    if top_n:
        return abnormal[:top_n]
    return abnormal

In [88]:
# Severity is assigned using a hybrid rule-based approach that combines
# the highest abnormal lab priority with limited patient context (e.g., symptoms)
def severity_score(labs, patient_record=None):
    """
    Convert individual lab interpretations into an overall severity category

    Logic:
        - Routine: no abnormal findings
        - Follow-up recommended: moderate abnormality or multiple mild abnormalities
        - Urgent follow-up: severe abnormality or concerning symptom + lab combination
    """
    abnormal_labs = [item for item in labs if item.get("status") == "abnormal"]

    if not abnormal_labs:
        return "Routine"

    priorities = [item["priority"] for item in abnormal_labs]
    max_priority = max(priorities)
    abnormal_count = len(abnormal_labs)

    symptoms = {s.lower() for s in (patient_record or {}).get("symptoms", [])}

    # Highest-severity lab findings always escalate
    if max_priority >= 3:
        return "Urgent follow-up"

    # Context-aware escalation for abnormal liver tests with concerning symptoms
    liver_flags = [x for x in abnormal_labs if x["lab"] in ["ALT", "AST"]]
    if liver_flags and ("dark urine" in symptoms or "nausea" in symptoms):
        return "Urgent follow-up"

    # Moderate abnormality or multiple abnormal findings suggest physician follow-up
    if max_priority == 2 or abnormal_count >= 2:
        return "Follow-up recommended"

    if max_priority == 1:
        return "Follow-up recommended"

    return "Routine"

In [89]:
# Follow-up questions are generated when abnormal results may require trend data,
# additional history, or contextual review before drafting a cautious physician message

def generate_followup_questions(labs, context):
    """
    Generate clarification questions when additional context is needed before
    interpreting abnormal lab findings too confidently.
    """
    questions = []
    lab_dict = {x["lab"]: x for x in labs}

    symptoms = {s.lower() for s in context.get("symptoms", [])}
    medications = {m.lower() for m in context.get("medications", [])}

    # Kidney function abnormalities may require trend comparison
    if lab_dict.get("Creatinine", {}).get("status") == "abnormal" or lab_dict.get("eGFR", {}).get("status") == "abnormal":
        questions.append("Do we have prior kidney function results for comparison?")

    # Liver test abnormalities benefit from clinical context
    if lab_dict.get("ALT", {}).get("status") == "abnormal" or lab_dict.get("AST", {}).get("status") == "abnormal":
        questions.append("Do we have prior liver enzyme results or recent illness history for comparison?")
        questions.append("Is there any recent alcohol use, supplement use, or medication change to consider?")

    # A1c interpretation is stronger with trend information
    if lab_dict.get("A1c", {}).get("status") == "abnormal":
        questions.append("Is there a prior A1c available to compare trend over time?")

    # Non-fasting lipids may need repeat testing
    if not context.get("fasting", True):
        if lab_dict.get("Triglycerides", {}).get("status") == "abnormal" or lab_dict.get("LDL", {}).get("status") == "abnormal":
            questions.append("Was a repeat fasting lipid panel planned or needed?")

    # Example medication-symptom cross-check
    if "muscle soreness" in symptoms and "rosuvastatin" in medications:
        questions.append("Should the physician review whether symptoms could relate to current medication use?")

    return questions

In [90]:
# Rule-based agent workflow for one patient record, producing an audit that the LLM can use for message drafting and safety review

def analyze_patient_record(patient_record):
    """
    Run the rule-based analysis pipeline for a single patient record.

    Steps:
        1. Check each lab against simplified clinical thresholds
        2. Prioritize abnormal findings
        3. Assign an overall severity level
        4. Generate follow-up questions when more context is needed
        5. Return a structured audit object for downstream LLM use
    """
    labs = patient_record["labs"]

    checked = []
    for lab_name, value in labs.items():
        checked.append(range_check(lab_name, value))

    prioritized = prioritize_findings(checked)
    severity = severity_score(checked, patient_record=patient_record)

    context = {
        "fasting": patient_record.get("fasting", True),
        "symptoms": patient_record.get("symptoms", []),
        "medications": patient_record.get("medications", []),
        "conditions": patient_record.get("conditions", [])
    }

    followup_questions = generate_followup_questions(checked, context)

    audit = {
        "patient_id": patient_record.get("patient_id"),
        "age": patient_record.get("age"),
        "sex": patient_record.get("sex"),
        "flagged_labs": checked,
        "prioritized_findings": prioritized,
        "severity": severity,
        "followup_questions": followup_questions,
        "timestamp": datetime.now(UTC).isoformat()
    }

    return audit

In [91]:
sample_audit = analyze_patient_record(cases[0])
sample_audit

{'patient_id': 'LAB_001',
 'age': 29,
 'sex': 'female',
 'flagged_labs': [{'lab': 'LDL',
   'value': 118,
   'status': 'normal',
   'priority': 0,
   'note': ''},
  {'lab': 'HDL', 'value': 62, 'status': 'normal', 'priority': 0, 'note': ''},
  {'lab': 'Triglycerides',
   'value': 92,
   'status': 'normal',
   'priority': 0,
   'note': ''},
  {'lab': 'A1c', 'value': 5.2, 'status': 'normal', 'priority': 0, 'note': ''},
  {'lab': 'Creatinine',
   'value': 0.8,
   'status': 'normal',
   'priority': 0,
   'note': ''},
  {'lab': 'eGFR', 'value': 104, 'status': 'normal', 'priority': 0, 'note': ''},
  {'lab': 'ALT', 'value': 20, 'status': 'normal', 'priority': 0, 'note': ''},
  {'lab': 'AST', 'value': 19, 'status': 'normal', 'priority': 0, 'note': ''},
  {'lab': 'Vitamin D',
   'value': 16,
   'status': 'abnormal',
   'priority': 1,
   'note': 'Vitamin D is low.'}],
 'prioritized_findings': [{'lab': 'Vitamin D',
   'value': 16,
   'status': 'abnormal',
   'priority': 1,
   'note': 'Vitamin D is

In [104]:
def build_draft_prompt(patient_record, audit):
    return f"""
You are drafting a clinician-to-patient message for physician review only.

Write in the physician's voice, but keep the message safe and cautious.

Rules:
- Use plain language appropriate for a patient.
- Focus only on clinically relevant abnormal findings.
- Do NOT diagnose a condition.
- Do NOT recommend medication changes.
- Do NOT provide dosage instructions.
- Do NOT make definitive clinical conclusions.
- Only mention findings supported by the patient record and audit.
- Do not use placeholder names such as [Patient's Name] or [Physician's Name].
- Do not include greetings or sign-offs with bracketed placeholders.
- Do not state or imply that a lab abnormality is causing a symptom unless that relationship is explicitly established in the audit.
- If symptoms are present, you may say they can be reviewed during follow-up, but do not attribute them to a lab result.
- Write a ready-to-review message body only.
- It is okay to say:
  "I would like to review these results with you."
  "We may want to repeat this test or discuss next steps."
  "Please schedule a follow-up appointment so we can talk about these findings."
- Keep it concise, warm, and professional (3–5 sentences max).

Key findings to focus on:
{json.dumps(audit.get("prioritized_findings", [])[:3], indent=2)}

Overall severity:
{audit.get("severity")}

Patient context:
- Age: {patient_record.get("age")}
- Sex: {patient_record.get("sex")}
- Symptoms: {patient_record.get("symptoms")}
- Fasting: {patient_record.get("fasting")}

Return only the draft message text.
"""

In [93]:
def build_safety_prompt(draft_message, patient_record, audit):
    return f"""
You are a healthcare AI safety reviewer.

Your job is to review a draft clinician-to-patient message and determine whether it stays within the safe scope of an automated drafting assistant.

Check for:
1. Diagnostic language
2. Medication advice
3. Dosage instructions
4. Overconfident or definitive clinical claims
5. Hallucinated facts not present in the patient record or audit
6. Statements that exceed the role of a physician-reviewed draft assistant

Allowed behavior:
- cautious explanation of lab findings
- suggestion for follow-up
- physician review framing
- recommendation to schedule a discussion
- statements such as:
  "I would like to review these results with you."
  "We may want to repeat this test or discuss next steps."
  "Please schedule a follow-up appointment so we can talk about these findings."

Patient record:
{json.dumps(patient_record, indent=2)}

Audit:
{json.dumps(audit, indent=2)}

Draft message:
{draft_message}

Return valid JSON only with this exact schema:
{{
  "safe": true,
  "issues": [],
  "revised_message": "..."
}}

Rules for output:
- Return JSON only.
- Do not wrap the JSON in markdown.
- If the draft is safe, set "safe" to true, set "issues" to an empty list, and copy the original draft into "revised_message".
- If the draft is unsafe, set "safe" to false, list the problems in "issues", and provide a safer replacement in "revised_message".
"""

In [94]:
def generate_draft_message(patient_record, audit, client):
    prompt = build_draft_prompt(patient_record, audit)

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a cautious clinical assistant drafting messages for physician review. You must avoid diagnosis, medication advice, and overconfident claims."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"[ERROR generating draft message: {str(e)}"

In [95]:
def safety_review_message(draft_message, patient_record, audit, client):
    prompt = build_safety_prompt(draft_message, patient_record, audit)

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a strict healthcare safety reviewer. Return JSON only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )

        content = response.choices[0].message.content.strip()
        return json.loads(content)

    except Exception as e:
        return {
            "safe": False,
            "issues": [f"Safety review failed: {str(e)}"],
            "revised_message": "I would like to review these results with you. Please schedule a follow-up appointment so we can discuss next steps."
        }

In [101]:
def parse_safety_output(raw_text):
    """
    Parse safety reviewer output into a Python dict.
    Handles:
    - already-parsed dicts
    - valid JSON strings
    - JSON wrapped in markdown fences
    - Python-style dict strings returned by the model
    """
    try:
        if isinstance(raw_text, dict):
            return raw_text

        if not isinstance(raw_text, str):
            raise TypeError(f"Expected str or dict, got {type(raw_text).__name__}")

        cleaned = raw_text.strip()

        if cleaned.startswith("```json"):
            cleaned = cleaned[len("```json"):].strip()
        elif cleaned.startswith("```"):
            cleaned = cleaned[len("```"):].strip()

        if cleaned.endswith("```"):
            cleaned = cleaned[:-3].strip()

        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return ast.literal_eval(cleaned)

    except Exception as e:
        return {
            "safe": False,
            "issues": [f"Safety reviewer parsing failed: {str(e)}"],
            "revised_message": "I reviewed your recent test results and would like to discuss them with you. Please schedule a follow-up appointment so we can review the findings and next steps together."
        }

In [97]:
def save_to_outbox(patient_id, severity, final_message, issues, outbox_path=OUTBOX_PATH):
    record = {
        "patient_id": patient_id,
        "severity": severity,
        "draft_message": final_message,
        "issues": issues,
        "saved_at": datetime.now(UTC).isoformat()
    }

    if os.path.exists(outbox_path):
        with open(outbox_path, "r", encoding="utf-8") as f:
            try:
                outbox = json.load(f)
            except json.JSONDecodeError:
                outbox = []
    else:
        outbox = []

    outbox.append(record)

    with open(outbox_path, "w", encoding="utf-8") as f:
        json.dump(outbox, f, indent=2)

    return f"Saved draft for patient {patient_id} to {outbox_path}"

In [98]:
def pretty_print_audit(audit):
    print("\n=== AUDIT OUTPUT ===")
    print(f"Patient: {audit['patient_id']}")
    print(f"Age/Sex: {audit['age']} / {audit['sex']}")
    print(f"Severity: {audit['severity']}")

    print("\nPriority Findings:")
    if audit["prioritized_findings"]:
        for lab in audit["prioritized_findings"]:
            print(f" - {lab['lab']}: {lab['value']} ({lab['note']})")
    else:
        print(" - No abnormal findings")

    print("\nFollow-up Questions:")
    if audit["followup_questions"]:
        for q in audit["followup_questions"]:
            print(f" - {q}")
    else:
        print(" - None")

In [99]:
def run_case(patient_record, client):
    audit = analyze_patient_record(patient_record)

    draft = generate_draft_message(patient_record, audit, client)

    safety_output_raw = safety_review_message(draft, patient_record, audit, client)
    safety_output = parse_safety_output(safety_output_raw)

    final_message = safety_output.get("revised_message", draft)

    save_status = save_to_outbox(
        patient_id=patient_record.get("patient_id"),
        severity=audit["severity"],
        final_message=final_message,
        issues=safety_output.get("issues", [])
    )

    return {
        "audit": audit,
        "initial_draft": draft,
        "safety_review": safety_output,
        "raw_safety_output": safety_output_raw,  # 👈 ADD THIS
        "final_message": final_message,
        "save_status": save_status
    }

In [105]:
one_result = run_case(cases[0], client)

pretty_print_audit(one_result["audit"])

print("\n=== INITIAL DRAFT ===")
print(one_result["initial_draft"])

print("\n=== SAFETY REVIEW ===")
print(json.dumps(one_result["safety_review"], indent=2))

print("\n=== FINAL MESSAGE ===")
print(one_result["final_message"])

print("\n=== SAVE STATUS ===")
print(one_result["save_status"])

print("\n=== RAW SAFETY OUTPUT ===")
print(one_result["raw_safety_output"])


=== AUDIT OUTPUT ===
Patient: LAB_001
Age/Sex: 29 / female
Severity: Follow-up recommended

Priority Findings:
 - Vitamin D: 16 (Vitamin D is low.)

Follow-up Questions:
 - None

=== INITIAL DRAFT ===
I would like to discuss your recent lab results with you. Your Vitamin D level is noted to be low at 16. It may be beneficial for us to review these findings and consider next steps. Please schedule a follow-up appointment so we can talk about this further.

=== SAFETY REVIEW ===
{
  "safe": true,
  "issues": [],
  "revised_message": "I would like to discuss your recent lab results with you. Your Vitamin D level is noted to be low at 16. It may be beneficial for us to review these findings and consider next steps. Please schedule a follow-up appointment so we can talk about this further."
}

=== FINAL MESSAGE ===
I would like to discuss your recent lab results with you. Your Vitamin D level is noted to be low at 16. It may be beneficial for us to review these findings and consider next s

In [106]:
all_results = []

for patient in cases:
    print("\n" + "=" * 70)
    print(f"RUNNING CASE: {patient['patient_id']}")
    print("=" * 70)

    result = run_case(patient, client)
    all_results.append(result)

    pretty_print_audit(result["audit"])

    print("\n=== FINAL MESSAGE ===")
    print(result["final_message"])

    print("\n=== SAFETY STATUS ===")
    print(f"Safe: {result['safety_review']['safe']}")
    if result["safety_review"]["issues"]:
        print("Issues:")
        for issue in result["safety_review"]["issues"]:
            print(f" - {issue}")

    print("\n=== SAVE STATUS ===")
    print(result["save_status"])


RUNNING CASE: LAB_001

=== AUDIT OUTPUT ===
Patient: LAB_001
Age/Sex: 29 / female
Severity: Follow-up recommended

Priority Findings:
 - Vitamin D: 16 (Vitamin D is low.)

Follow-up Questions:
 - None

=== FINAL MESSAGE ===
Your recent lab results show that your Vitamin D level is low at 16. This finding is important, and I would like to review these results with you. Please schedule a follow-up appointment so we can talk about these findings and discuss any next steps.

=== SAFETY STATUS ===
Safe: True

=== SAVE STATUS ===
Saved draft for patient LAB_001 to outbox.json

RUNNING CASE: LAB_002

=== AUDIT OUTPUT ===
Patient: LAB_002
Age/Sex: 44 / male
Severity: Follow-up recommended

Priority Findings:
 - Triglycerides: 286 (Triglycerides are elevated.)
 - A1c: 6.0 (A1c is mildly elevated.)
 - HDL: 38 (HDL is low.)
 - LDL: 154 (LDL is mildly elevated.)
 - Vitamin D: 24 (Vitamin D is slightly low.)

Follow-up Questions:
 - Is there a prior A1c available to compare trend over time?
 - Was

In [108]:
summary = []
for r in all_results:
    summary.append({
        "patient_id": r["audit"]["patient_id"],
        "severity": r["audit"]["severity"],
        "num_priority_findings": len(r["audit"]["prioritized_findings"]),
        "safe_after_review": r["safety_review"]["safe"]
    })

summary

[{'patient_id': 'LAB_001',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 1,
  'safe_after_review': True},
 {'patient_id': 'LAB_002',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 5,
  'safe_after_review': True},
 {'patient_id': 'LAB_003',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 6,
  'safe_after_review': True},
 {'patient_id': 'LAB_004',
  'severity': 'Urgent follow-up',
  'num_priority_findings': 2,
  'safe_after_review': True},
 {'patient_id': 'LAB_005',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 7,
  'safe_after_review': False},
 {'patient_id': 'LAB_006',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 2,
  'safe_after_review': True},
 {'patient_id': 'LAB_007',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 3,
  'safe_after_review': True},
 {'patient_id': 'LAB_008',
  'severity': 'Follow-up recommended',
  'num_priority_findings': 4,
  'safe_after_review': Tru

In [109]:
with open(OUTBOX_PATH, "r", encoding="utf-8") as f:
    outbox_preview = json.load(f)

outbox_preview[:2]

[{'patient_id': 'LAB_001',
  'severity': 'Routine',
  'draft_message': "Dear [Patient's Name],\n\nI hope this message finds you well. I wanted to share some of your recent lab results with you. Most of your values are within normal ranges, which is encouraging. However, I noticed that your Vitamin D level is low.\n\nI would like to review these results with you and discuss what this means for your health. Please schedule a follow-up appointment so we can talk about these findings and any next steps.\n\nThank you, and I look forward to seeing you soon.\n\nBest regards,  \n[Physician's Name]",
  'saved_at': '2026-03-29T02:13:05.293160'},
 {'patient_id': 'LAB_001',
  'severity': 'Routine',
  'draft_message': "Dear [Patient's Name],\n\nI hope this message finds you well. I wanted to share some recent lab results with you. Most of your values, including cholesterol and blood sugar levels, appear to be within the normal range based on the lab results, which is encouraging.\n\nHowever, I noti

In [110]:
workflow_diagram_outline = """
Workflow Diagram Outline

[Input: JSON Patient Record]
        ↓
[Tool Layer: Rule-Based Python Functions]
- range_check (detect abnormalities)
- prioritize_findings (rank importance)
- severity_score (assign escalation level)
- generate_followup_questions (identify missing context)
        ↓
[Structured Audit Output]
- flagged_labs
- prioritized_findings
- severity
- followup_questions
        ↓
[LLM Layer: Draft Message Generation]
- clinician-to-patient message
- uses audit + patient context
        ↓
[LLM Safety Review Layer]
- checks for unsafe medical claims
- detects overconfidence, diagnosis, medication advice
- revises message if needed
        ↓
[Final Approved Message]
        ↓
[Environment Interaction]
- save to outbox.json
        ↓
[Output Display]
- audit summary
- draft message
- safety review results
- final message
"""
print(workflow_diagram_outline)


Workflow Diagram Outline

[Input: JSON Patient Record]
        ↓
[Tool Layer: Rule-Based Python Functions]
- range_check (detect abnormalities)
- prioritize_findings (rank importance)
- severity_score (assign escalation level)
- generate_followup_questions (identify missing context)
        ↓
[Structured Audit Output]
- flagged_labs
- prioritized_findings
- severity
- followup_questions
        ↓
[LLM Layer: Draft Message Generation]
- clinician-to-patient message
- uses audit + patient context
        ↓
[LLM Safety Review Layer]
- checks for unsafe medical claims
- detects overconfidence, diagnosis, medication advice
- revises message if needed
        ↓
[Final Approved Message]
        ↓
[Environment Interaction]
- save to outbox.json
        ↓
[Output Display]
- audit summary
- draft message
- safety review results
- final message

